In [27]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import torch.nn as nn
import random
from dataclasses import dataclass
import math
from torch.nn import functional as F

text = Path('corpus/prose_20M.txt').read_text(encoding='utf-8')
text[:40]

'Տպագրվում է ՀԽՍՀ ԳԱ Մ. Աբեղյանի անվան գր'

In [28]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.qkv = nn.Linear(self.config.n_embed, 3*self.config.n_embed)
        self.c_proj = nn.Linear(self.config.n_embed, self.config.n_embed)
        self.register_buffer('bias', torch.tril(torch.ones(1, 1, self.config.block_size, self.config.block_size)))

    def forward(self, x):
        assert((self.config.n_embed % self.config.n_head) == 0)
        B,T,C = x.shape
        qkv = self.qkv(x)
        q,k,v = qkv.split(self.config.n_embed, dim=-1)
        Q = q.view(B,T, self.config.n_head, self.config.n_embed//self.config.n_head).transpose(1,2)  # B, nh, T, hs
        K = k.view(B,T, self.config.n_head, self.config.n_embed//self.config.n_head).transpose(1,2)  # B, nh, T, hs
        V = v.view(B,T, self.config.n_head, self.config.n_embed//self.config.n_head).transpose(1,2)  # B, nh, T, hs
        attn = Q @ K.transpose(-1, -2) / math.sqrt(K.shape[-1])  # B, nh, T, T
        attn = attn.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        y = attn @ V        # (B, nh, T, T) @ (B, nh, T, hs)  -> B, nh, T, hs
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y 

In [29]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embed, 4*config.n_embed)
        self.gelu = nn.GELU(approximate='tanh')
        self.proj = nn.Linear(4*config.n_embed, config.n_embed)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.proj(x)
        return x


In [30]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lf1 = nn.LayerNorm(config.n_embed)
        self.attn = CausalSelfAttention(config)       
        self.lf2 = nn.LayerNorm(config.n_embed)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.lf1(x))
        x = x + self.mlp(self.lf2(x))
        return x



In [31]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embed),
            pte = nn.Embedding(config.block_size, config.n_embed),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embed)
        ))

        self.lm_head = nn.Linear(config.n_embed, config.vocab_size)

    def forward(self, idx):
        B,T = idx.shape
        tok_embed = self.transformer['wte'](idx)
        pos_embed = self.transformer['pte'](torch.arange(T, device=idx.device))
        x = tok_embed + pos_embed
        for block in self.transformer['h']:
            x = block(x)
        logits = self.transformer['ln_f'](x)
        logits = self.lm_head(logits)
        return logits

       

In [32]:
@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 8192
    n_head: int = 12
    n_embed: int = 384
    n_layer: int = 12

config = GPTConfig()

In [33]:
config = GPTConfig()
x = torch.randn(2, 8, 384)
c_attn = MLP(config)
c_attn(x)

tensor([[[-0.2306, -0.2115,  0.0709,  ..., -0.2802,  0.1475,  0.0534],
         [-0.3565,  0.1422,  0.0298,  ...,  0.1909, -0.0364, -0.4281],
         [-0.1005,  0.2313, -0.4190,  ..., -0.1826,  0.1659, -0.3450],
         ...,
         [-0.1652, -0.0667, -0.4142,  ...,  0.2647, -0.0956,  0.0026],
         [ 0.1420, -0.1973,  0.0978,  ..., -0.0129,  0.1423, -0.0938],
         [-0.1838, -0.2199,  0.0340,  ..., -0.0123, -0.2640, -0.1700]],

        [[ 0.2760, -0.3608, -0.1041,  ..., -0.0387,  0.4169, -0.1355],
         [-0.2428, -0.2027,  0.1180,  ..., -0.3599,  0.1491,  0.1544],
         [-0.1211, -0.1135,  0.0681,  ..., -0.1673,  0.1055, -0.1673],
         ...,
         [-0.1675, -0.1740,  0.2412,  ..., -0.2217,  0.2406, -0.0065],
         [-0.1692, -0.3378,  0.2720,  ..., -0.1776,  0.0564, -0.1745],
         [-0.0523,  0.0208, -0.0493,  ..., -0.0255, -0.1879,  0.1631]]],
       grad_fn=<ViewBackward0>)